In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Trisha Tyagi\\OneDrive\\Desktop\\Kidney-disease-classification\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\Trisha Tyagi\\OneDrive\\Desktop\\Kidney-disease-classification'

In [ ]:
import mlflow

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/Trisha-Tyagi/Kidney-disease-classification.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "Trisha-Tyagi"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "YOUR_PASSWORD" # enter you own token password here

In [9]:
import tensorflow as tf

In [10]:
model=tf.keras.models.load_model("artifacts/training/model.h5")

In [11]:

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [12]:

from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [17]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/kidney_ct_scan",
            mlflow_uri="https://dagshub.com/Trisha-Tyagi/Kidney-disease-classification.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [18]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [19]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )
            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model, "model")



In [20]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

[2026-08-11 12:27:04,157: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-11 12:27:04,159: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-11 12:27:04,161: INFO: common: created directory at: artifacts]
Found 2207 images belonging to 2 classes.
138/138 [==============================] - 823s 6s/step - loss: 1.2227 - accuracy: 0.8695
[2026-08-11 12:40:48,165: INFO: common: json file saved at: scores.json]


2026/08/11 12:40:52 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-08-11 12:40:53,676: WARNING: save: Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 14). These functions will not be directly callable after loading.]
INFO:tensorflow:Assets written to: C:\Users\TRISHA~1\AppData\Local\Temp\tmpa_tuu8rq\model\data\model\assets
[2026-08-11 12:40:54,190: INFO: builder_impl: Assets written to: C:\Users\TRISHA~1\AppData\Local\Temp\tmpa_tuu8rq\model\data\model\assets]


2026/08/11 12:41:05 WARNING mlflow.utils.requirements_utils: The following packages were not found in the public PyPI package index as of 2023-02-28; if these packages are not present in the public PyPI index, you must install them manually before loading your model: {'backports-tarfile'}
c:\Users\Trisha Tyagi\anaconda3\envs\kidney310\lib\site-packages\_distutils_hack\__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


[2026-08-11 12:44:43,332: WARNING: connectionpool: Retrying (Retry(total=4, connect=5, read=4, redirect=5, status=5)) after connection broken by 'ConnectionAbortedError(10053, 'An established connection was aborted by the software in your host machine')': /Trisha-Tyagi/Kidney-disease-classification.mlflow/api/2.0/mlflow-artifacts/artifacts/d4dfef90d7b94be3abfd194130249813/d609e0ec09084ce9b5fde4f4c4fd2c7c/artifacts/model/data/model/variables/variables.data-00000-of-00001]
[2026-08-11 12:46:12,846: WARNING: connectionpool: Retrying (Retry(total=3, connect=5, read=3, redirect=5, status=5)) after connection broken by 'ConnectionAbortedError(10053, 'An established connection was aborted by the software in your host machine')': /Trisha-Tyagi/Kidney-disease-classification.mlflow/api/2.0/mlflow-artifacts/artifacts/d4dfef90d7b94be3abfd194130249813/d609e0ec09084ce9b5fde4f4c4fd2c7c/artifacts/model/data/model/variables/variables.data-00000-of-00001]
[2026-08-11 12:46:39,987: WARNING: connectionpoo

Successfully registered model 'VGG16Model'.
2026/08/11 12:53:19 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: VGG16Model, version 1
Created version '1' of model 'VGG16Model'.
